# LIPAS Train4a GO-NIGHT

Beat train2 held-out RMSE 4.9685. Channel-normalized Huber, solar+rad priority, all blend > 0.
Kaggle GPU only. No CPU fallback.


In [ ]:
import os, shutil, sys
from pathlib import Path
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
os.environ.setdefault('MPLCONFIGDIR', '/tmp/mpl')
os.environ.setdefault('TF_DETERMINISTIC_OPS', '1')
WORKDIR = Path('/kaggle/working/lipas')
WORKDIR.mkdir(parents=True, exist_ok=True)
CSV_NAME = 'train7_training_locked_y_ext_drivers.csv'
root = None
for base in [Path('/kaggle/input'), Path('/kaggle/working'), Path('.')]:
    if not base.exists():
        continue
    hits = list(base.rglob('train4_kaggle.py'))
    if hits:
        root = hits[0].parent
        break
assert root is not None, 'Attach lukatakki/lipas-train4 dataset'
for item in root.iterdir():
    dest = WORKDIR / item.name
    if item.is_dir():
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(item, dest, dirs_exist_ok=True)
    else:
        shutil.copy2(item, dest)
(WORKDIR / 'configs').mkdir(exist_ok=True)
(WORKDIR / 'scripts').mkdir(exist_ok=True)
(WORKDIR / 'data' / 'ingested_extended').mkdir(parents=True, exist_ok=True)
for name in ('train4a.json', 'train4b.json', 'train4c.json', 'hybrid_blend_policy.json'):
    src = WORKDIR / name
    if src.exists():
        shutil.copy2(src, WORKDIR / 'configs' / name)
for name in ('train4_kaggle.py', 'fit_blend_and_calibration.py', 'evaluate.py'):
    src = WORKDIR / name
    if src.exists():
        shutil.copy2(src, WORKDIR / 'scripts' / name)
csv_dest = WORKDIR / 'data' / 'ingested_extended' / CSV_NAME
if not csv_dest.exists():
    cands = list(WORKDIR.rglob(CSV_NAME))
    assert cands, f'Missing {CSV_NAME}'
    shutil.copy2(cands[0], csv_dest)
os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))
import tensorflow as tf
print('ROOT', WORKDIR.resolve())
print('TF', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs', gpus)
assert gpus, 'BLOCKER: no GPU — refuse CPU fallback'


In [ ]:
import subprocess, sys
from pathlib import Path
out_dir = Path('/kaggle/working/saved_model_train4a')
cmd = [
    sys.executable, 'scripts/train4_kaggle.py',
    '--stage', 'cheap+t4a',
    '--model-dir', str(out_dir),
    '--config', 'configs/train4a.json',
    '--csv', 'data/ingested_extended/train7_training_locked_y_ext_drivers.csv',
]
print('Running:', ' '.join(cmd))
rc = subprocess.call(cmd)
print('exit', rc)
assert rc == 0, 'Train4a failed'
print('Artifacts:', sorted(p.name for p in out_dir.iterdir()) if out_dir.exists() else None)
cheap = Path('/kaggle/working/cheap_checks/cheap_checks.json')
print('cheap_checks exists', cheap.exists())
